# Counting NMNH types

1. In EMu, search all departments in Catalog for Type Status = `\+`
2. Export matching records using IDSC_TypeStatuses to the source folder
3. Run notebook

You can also include estimates of the number of types in department backlogs by
adding that data to *source/backlog.csv*.

The notebook includes six constants that control some aspects of how the type data is
aggreagated. See the next cell for more information about those values.

In [ ]:
import re
from itertools import permutations

import pandas as pd

from xmu import EMuReader, EMuRecord


pd.set_option("display.max_rows", 500)

INCLUDE_BACKLOG = False  # include backlog estimates
INCLUDE_COMPOUND = True  # include unofficial compound statuses
INCLUDE_MODIFIED = True  # include statuses with modifying keywords
INCLUDE_PARTS = True  # includes type parts and fragments
INCLUDE_UNCERTAIN = True  # include uncertain
INCLUDE_MINERALS = True  # include mineral types

In [ ]:
def remove_modified(val: str) -> str:
    """Remove modifiers from value"""
    val = re.split(r"\bof\b", val)[0]
    return (
        re.sub(r"\b(Collection|Conserved|Primary|Material)\b", "", val, flags=re.I)
        .strip()
        .lower()
    )


def remove_uncertain(val: str) -> str:
    """Remove uncertainty markers from value"""
    val = remove_modified(val)
    return (
        re.sub(r"\b(Possible|Probable|Uncertain|Unconfirmed)\b", "", val, flags=re.I)
        .rstrip("s? ")
        .strip()
        .lower()
    )


def remove_part(val: str) -> str:
    """Remove part modifiers from value"""
    return (
        re.sub(r"\b(Part Of|Fragment)\b", "", val, flags=re.I)
        .rstrip("s? ")
        .strip()
        .lower()
    )


def get_prefixes(vals: list | tuple) -> list[str]:
    """Get the list of prefixes for a list of statuses"""
    return [s.replace("type", "") for s in vals]


def get_compounds(vals: list | tuple, n: int = 2, m: int = 3) -> list[str]:
    """Get all possible compound statuses based on prefixes in statuses"""
    prefixes = get_prefixes(vals)
    compounds = []
    for i in range(n, m + 1):
        compounds.extend(["".join(p) + "type" for p in permutations(prefixes, i)])
    return compounds


def map_status(types: pd.DataFrame, val: str, coll: str) -> list[dict]:
    """Map status to recognized value"""

    # Split status if multiple
    vals = re.split(r" *(?:\+|\band\b|;) *", val)
    if len(vals) > 1:
        statuses = []
        for val in vals:
            statuses.extend(map_status(types, val, coll))
        return statuses

    val = val.lower()

    # Map coll to subsets
    subsets = ["all"]
    if coll == "Botany":
        subsets.append("plants")
    else:
        subsets.append("animals")

    subset = types[types["Subset"].isin(subsets)]
    statuses = set(subset["Status"])

    if val in statuses:
        mapped = val
        match = "exact"
    elif val.rstrip("s ") in statuses:
        mapped = val.rstrip("s ")
        match = "plural"
    elif remove_modified(val) in statuses:
        mapped = remove_modified(val)
        match = "modified"
    elif remove_uncertain(val) in statuses:
        mapped = remove_uncertain(val)
        match = "uncertain"
    elif remove_part(val) in statuses:
        mapped = remove_part(val)
        match = "part"
    elif val in get_compounds(subset["Status"]):
        mapped = val
        match = "compound"
    else:
        mapped = "--"
        match = "--"

    return [
        {"collection": coll, "value": val, "mapped": mapped, "match": match, "count": 1}
    ]

In [ ]:
# Read and prep type lookup
types = pd.read_csv("source/types.csv")

# Terms that appear in both ICZN and ICN are interepreted the same for all departments
cond = (types["ICZN"] == "Yes") & (types["ICN"] == "Yes")
types.loc[cond, "Subset"] = "all"

# Terms that only appear in ICN only count for plants
cond = (types["ICN"] == "Yes") & pd.isna(types["Subset"])
types.loc[cond, "Subset"] = "plants"

# Terms that only appear in ICZN only count for animals
cond = (types["ICZN"] == "Yes") & pd.isna(types["Subset"])
types.loc[cond, "Subset"] = "animals"

# Old and generic terms still count. Note that ento considers type unreliable.
# That is addressed below.
cond = (types["Status"].isin(("type", "cotype"))) & pd.isna(types["Subset"])
types.loc[cond, "Subset"] = "all"

# Unofficial statuses count if marked primary or secondary. This is just allotype
# for now.
cond = (types["Kind"].isin(("primary", "secondary"))) & pd.isna(types["Subset"])
types.loc[cond, "Subset"] = "all"

types

In [ ]:
# Read data from EMu export
def callback(path: str) -> list:
    """Parses data from EMu export file"""
    reader = EMuReader(path, rec_class=EMuRecord)
    rows = []
    for rec in reader:

        statuses = rec.get("CitTypeStatus_tab", [])

        # Collection is used to determine the valid type statuses
        coll = rec["CatDepartment"]
        if coll == "Vertebrate Zoology":
            coll = "VZ-" + rec["CatDivision"]
            if coll.startswith("VZ-A"):
                coll = "VZ-Herps"

        # HACK: Mark Ento Inquire Types as uncertain
        elif (
            rec["CatDepartment"] == "Entomology"
            and rec["CatCatalog"] == "Inquire Types"
        ):
            statuses = [s.rstrip("?") + "?" for s in statuses]

        for status in statuses:
            rows.extend(map_status(types, status, coll))

    return rows


rows = EMuReader("source/xmldata.xml").from_xml_parallel(callback)

In [ ]:
# Add backlog estimates
if INCLUDE_BACKLOG:
    backlog = pd.read_csv("source/backlog.csv")
    backlog = backlog.rename(columns={c: c.lower() for c in backlog.columns})
    backlog["value"] = "type"
    backlog["mapped"] = "type"
    backlog["match"] = "backlog"
    rows.extend(backlog.to_dict("records"))

In [ ]:
# Create type count dataframe
df = pd.DataFrame(rows)
agg = df.groupby(["collection", "value", "mapped", "match"]).sum("count").reset_index()
agg = agg.merge(types, how="left", left_on="mapped", right_on="Status")
agg = agg.rename(columns={c: c.lower() for c in agg.columns})
agg = agg[["collection", "value", "mapped", "match", "kind", "iczn", "icn", "count"]]

# Unofficial compound statuses are counted as secondary
cond = agg["match"] == "compound"
agg.loc[cond, "kind"] = "secondary"

# Calculate percentage
agg["percent"] = agg["count"] / agg["count"].sum()

agg_all = agg.copy()

if not INCLUDE_MODIFIED:
    agg = agg[agg["match"] != "modified"]

if not INCLUDE_PARTS:
    agg = agg[agg["match"] != "part"]

if not INCLUDE_UNCERTAIN:
    agg = agg[agg["match"] != "uncertain"]

if not INCLUDE_MINERALS:
    agg = agg[agg["collection"] != "Mineral Sciences"]


def add_total(df):
    """Adds total"""
    df["percent"] = df["count"] / df["count"].sum()
    total = pd.DataFrame([{df.columns[0]: "total", "count": df["count"].sum()}])
    return pd.concat([df, total])


all_types = add_total(agg.groupby("kind").sum()[["count"]].reset_index())
primary = add_total(
    agg[agg["kind"] == "primary"].groupby("collection").sum()[["count"]].reset_index()
)
secondary = add_total(
    agg[agg["kind"] == "secondary"].groupby("collection").sum()[["count"]].reset_index()
)
total = add_total(
    agg[~pd.isna(agg["kind"])].groupby("collection").sum()[["count"]].reset_index()
)
matches = add_total(agg_all.groupby("match").sum()[["count"]].reset_index())

sheets = {
    "counts": all_types,
    "total_by_dept": total,
    "primary_by_dept": primary,
    "secondary_by_dept": secondary,
    "data": agg_all,
    "match_types": matches,
}

In [ ]:
with pd.ExcelWriter("nmnh-type-counts.xlsx") as writer:
    for sheet_name, sheet in sheets.items():
        sheet.to_excel(writer, sheet_name=sheet_name, index=0)

In [ ]:
exact = agg_all[
    (agg_all["match"] == "exact")
    & ((agg_all["iczn"] == "Yes") | (agg_all["icn"] == "Yes"))
].sum()
print(f"{exact['percent']:.1%} of statuses in EMu exactly match an official status")